# Lab 08 · Neural networks

Explore neural networks by recognising small drawings and predicting a kettle's heating time.

Fill each `____` using the comments, then run the cells in order.


## Image Arrays


In [ ]:
# Use these package aliases and helper names.
import numpy as ____                     # np
import matplotlib.pyplot as ____       # plt
from tensorflow import ____          # keras
from sklearn.model_selection import ____  # train_test_split
from sklearn.preprocessing import ____  # MinMaxScaler
from pathlib import ____              # Path
keras.utils.set_random_seed(____)        # Seed 81.
rng = np.random.default_rng(____)        # Seed 81.


In [ ]:
# A 1 is a dark pen mark; a 0 is blank paper. Rows draw book, mug, spoon.
book = np.array([[1,1,1,1],[1,0,0,1],[1,0,0,1],[1,1,1,1]], dtype=____)  # Floating-point pixels.
mug = np.array([[1,1,1,0],[1,0,1,1],[1,0,1,1],[1,1,1,0]], dtype=____)   # Floating-point pixels.
spoon = np.array([[0,1,1,0],[0,1,1,0],[0,1,0,0],[0,1,0,0]], dtype=____) # Floating-point pixels.
outlines = np.stack([book, mug, ____])  # Keep book, mug, spoon order.
names = [____]  # Same label order as the drawings.
labels = np.repeat(np.arange(3), ____)     # 60 noisy versions per drawing.
images = np.clip(outlines[labels] + rng.normal(0, 0.12, (180, 4, 4)), 0, ____)  # Cap brightness at 1.
images = (images * ____).astype("uint8")  # Store pixels on a 0–255 scale.


In [ ]:
# Display the clean drawings with black marks on white paper.
fig, axes = plt.subplots(1, ____, figsize=(6, 2))  # Three panels.
for i in range(____):                     # Visit all three drawings.
    axes[i].imshow(outlines[i], cmap=____, vmin=0, vmax=1)  # Reversed grayscale: gray_r.
    axes[i].set_title(____)          # Label this panel with its object name.
    axes[i].axis(____)                # Hide axes with off.
plt.____()


In [ ]:
# Keep 20% for test; use 25% of the rest for validation. Flatten 4x4 to 16.
img_build, img_test, lab_build, lab_test = train_test_split(images, labels, test_size=____, stratify=labels, random_state=81)
img_train, img_val, lab_train, lab_val = train_test_split(img_build, lab_build, test_size=____, stratify=lab_build, random_state=81)
x_train = img_train.reshape(-1, ____).astype("float32") / 255
x_val = img_val.reshape(-1, ____).astype("float32") / 255
x_test = img_test.reshape(-1, ____).astype("float32") / 255
y_train = keras.utils.to_categorical(lab_train, num_classes=____)
y_val = keras.utils.to_categorical(lab_val, num_classes=____)
y_test = keras.utils.to_categorical(lab_test, num_classes=____)
print(x_train.shape, ____)


## Classification Neural Networks


In [ ]:
# Sixteen inputs, a small nonlinear layer, and three class probabilities.
classifier = keras.____([        # Sequential model.
    keras.Input(shape=(____,)),
    keras.layers.Dense(____, activation="relu"),  # 18 hidden units.
    keras.layers.Dropout(____),         # Dropout rate 0.15.
    keras.layers.Dense(____, activation="softmax")])
classifier.compile(optimizer="adam", loss=____, metrics=["accuracy"])  # Cross-entropy for one-hot targets.
classifier.____()


In [ ]:
# Short training; validation is monitored, not used for gradient updates.
class_history = classifier.fit(x_train, y_train, epochs=____, batch_size=18, validation_data=(x_val, y_val), verbose=0)  # 18 epochs.
plt.plot(class_history.history[____], label="train")
plt.plot(class_history.history[____], label="validation")
plt.xlabel(____)              # Horizontal label: Epoch index.
plt.ylabel(____)            # Vertical label: Cross-entropy.
plt.legend(loc=____)                 # Automatic legend location: best.
plt.____()


In [ ]:
# Evaluate on untouched drawings, then inspect one probability vector.
class_loss, class_accuracy = classifier.evaluate(____, y_test, verbose=0)
probabilities = classifier.predict(x_test[:1], verbose=____)[0]  # Silent prediction: verbose=0.
predicted_id = np.____(probabilities)
print(class_accuracy, probabilities, names[____])
print("Probability sum:", probabilities.____())


## Regression Neural Networks


In [ ]:
# Invent 240 kettle runs: [water litres, starting temperature in Celsius].
water = rng.uniform(0.3, 1.8, size=____)
start_temp = rng.uniform(10, 30, size=____)
features = np.column_stack([water, ____]).astype("float32")
minutes = (1 + 3 * water - 0.025 * start_temp + rng.normal(0, 0.08, 240)).astype(____)  # float32 targets.
# This invented relationship is for practice, not a physical calibration.


In [ ]:
# Keep 20% for test, then 25% of the rest for validation. Fit scaler on train only.
f_build, f_test, t_build, t_test = train_test_split(features, minutes, test_size=____, random_state=81)
f_train, f_val, t_train, t_val = train_test_split(f_build, t_build, test_size=____, random_state=81)
scaler = ____()
s_train = scaler.fit_transform(____)
s_val = scaler.transform(____)
s_test = scaler.transform(____)


In [ ]:
# Build a fresh regressor for each optimizer; one linear output predicts minutes.
def build_regressor(____):         # Parameter name: optimizer.
    model = keras.____([          # Sequential model.
        keras.Input(shape=(____,)),        # Water volume and starting temperature.
        keras.layers.Dense(____, activation="relu"),  # First layer: 12 units.
        keras.layers.Dense(____, activation="relu"),   # Second layer: 8 units.
        keras.layers.Dense(____)])
    model.compile(optimizer=____, loss="mean_squared_error", metrics=["mean_absolute_error"])  # Use the function argument.
    return ____


In [ ]:
# Compare Adam and SGD with the same starting seed, data, and epoch limit.
candidates, validation_errors = ____, ____  # Two empty dictionaries.
for optimizer_name in [____]:
    keras.utils.set_random_seed(____)   # Same starting seed: 105.
    candidate = build_regressor(____)
    stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=____, restore_best_weights=True)  # Patience: 4 epochs.
    history = candidate.fit(s_train, t_train, epochs=____, batch_size=24, validation_data=(s_val, t_val), callbacks=[stop], verbose=0)  # At most 30 epochs.
    candidates[optimizer_name] = ____
    validation_errors[optimizer_name] = candidate.evaluate(s_val, t_val, verbose=____)[1]  # Silent evaluation: 0.
    print(optimizer_name, validation_errors[____])


In [ ]:
# Choose using validation MAE; do not select a model using the test rows.
best_name = min(validation_errors, key=____)
best_model = candidates[____]
test_loss, test_mae = best_model.evaluate(____, t_test, verbose=0)
print(best_name, ____)


## Model Persistence


In [ ]:
# Save model plus learned scaling arrays in the runtime working folder.
runtime_dir = Path(____)  # Folder name: lab08_runtime.
runtime_dir.mkdir(exist_ok=____)
best_model.save(runtime_dir / ____)  # Model file: kettle.keras.
np.savez(runtime_dir / "scaling.npz", scale=scaler.____, offset=scaler.min_)
# Keep feature order: water litres, then starting temperature.


In [ ]:
# Reload your own files; reproduce test predictions before using a new example.
loaded_model = keras.models.load_model(runtime_dir / ____)  # Same model filename.
saved_scaling = np.load(runtime_dir / ____)  # Scaling file: scaling.npz.
reloaded_test = f_test * saved_scaling[____] + saved_scaling["offset"]
before = best_model.predict(s_test, verbose=____)  # Silent prediction: 0.
after = loaded_model.predict(reloaded_test, verbose=____)  # Silent prediction: 0.
print(np.allclose(before, after, atol=____))  # Absolute tolerance: 1e-6.
print("Reloaded MAE:", loaded_model.evaluate(reloaded_test, t_test, verbose=____)[1])  # Silent: 0.
new_run = np.array([[____]], dtype="float32")  # 0.8 litres, 25 Celsius.
scaled_run = new_run * saved_scaling[____] + saved_scaling["offset"]
print("Predicted minutes:", loaded_model.predict(scaled_run, verbose=____)[0, 0])  # Silent: 0.
saved_scaling.____()
